# TSBOW 원본 영상 인코딩 설정 확인

목적: 실제 CCTV 영상(TSBOW)이 어떤 설정으로 압축돼 있는지 확인해서, 우리 "CCTV형" 인코딩 설정의 근거로 쓰기

설정
1. **Private 노트북**으로 만들 것 (TSBOW 라이선스 CC BY-NC-ND)
2. Add-ons → Secrets → `HF_TOKEN` 연결 (코드에 토큰 쓰지 말 것)
3. Accelerator None / Internet On
4. Save & Run All

방식: videos.zip 전체를 받지 않고, Hugging Face에서 zip 안의 영상만 골라서 하나씩 받아 분석 후 바로 삭제함
(디스크 거의 안 씀)

확인 항목: 코덱·프로파일, 해상도·fps, B프레임 유무, GOP(키프레임 간격), 평균 비트레이트·bpp,
초당 비트레이트 변동(일정한지 = CBR 성격), 인코더 정보(파일에 남아 있으면 x264/x265 옵션)


In [ ]:
import os, io, re, json, random, shutil, zipfile, subprocess, collections
import numpy as np, pandas as pd
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, HfFileSystem, hf_hub_download

token = UserSecretsClient().get_secret("HF_TOKEN")
REPO = "SKKUAutoLab/TSBOW"
api = HfApi(token=token)
fs = HfFileSystem(token=token)

N_VIDEOS = 20        # 분석할 영상 수 (늘리면 시간 늘어남)
SEED = 0
TMP = "/tmp/tsbow_probe"; os.makedirs(TMP, exist_ok=True)
OUT = "/kaggle/working"

files = api.list_repo_files(REPO, repo_type="dataset")
vzips = [f for f in files if f.endswith("videos.zip")]
for p in api.get_paths_info(REPO, vzips, repo_type="dataset"):
    print(f"{p.path:60s} {p.size/1e9:7.2f} GB")
meta_csv = [f for f in files if f.endswith("TSBOW_info.csv")]
print("metadata:", meta_csv)

In [ ]:
# 메타데이터 (시나리오·날씨·규모별로 고르게 뽑기 위해)
meta = None
if meta_csv:
    p = hf_hub_download(REPO, meta_csv[0], repo_type="dataset", token=token, local_dir=TMP)
    meta = pd.read_csv(p)
    meta.columns = [c.strip().upper() for c in meta.columns]
    print(meta.shape); print(meta.head())
    for c in ["SCENARIO", "WEATHER", "SCALE", "DAYTIME", "ROADTYPE"]:
        if c in meta: print(c, dict(meta[c].value_counts()))

In [ ]:
# zip 안 영상 목록 (zip 전체를 받지 않고 목록만 읽음)
# 같은 영상이 두 버전에 중복될 수 있어서 v1.1.0 우선, 없으면 전체
ZIPS = [z for z in vzips if z.startswith("v1.1.0")] or vzips
pool = []
for z in ZIPS:
    f = fs.open(f"datasets/{REPO}/{z}", "rb", block_size=4 * 2**20)
    zf = zipfile.ZipFile(f)
    mp4 = [m for m in zf.infolist() if m.filename.lower().endswith((".mp4", ".mkv", ".avi", ".h264", ".h265"))
           and not os.path.basename(m.filename).startswith("._")]
    print(f"{z}: {len(mp4)}개, {sum(m.file_size for m in mp4)/1e9:.2f} GB, 압축방식 {collections.Counter(m.compress_type for m in mp4)}")
    pool += [(z, zf, m) for m in mp4]
print("전체 영상", len(pool))

In [ ]:
# 샘플 고르기: 메타데이터가 있으면 SCENARIO별로 고르게, 없으면 무작위
random.seed(SEED)
def vid(m): return os.path.splitext(os.path.basename(m.filename))[0]
if meta is not None and "VIDEO_ID" in meta and "SCENARIO" in meta:
    sc = dict(zip(meta["VIDEO_ID"].astype(str), meta["SCENARIO"].astype(str)))
    groups = collections.defaultdict(list)
    for t in pool: groups[sc.get(vid(t[2]), "?")].append(t)
    for g in groups.values(): random.shuffle(g)
    sample = []
    while len(sample) < min(N_VIDEOS, len(pool)):
        for g in groups.values():
            if g and len(sample) < N_VIDEOS: sample.append(g.pop())
else:
    sample = random.sample(pool, min(N_VIDEOS, len(pool)))
print([vid(t[2]) for t in sample])

In [ ]:
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode: raise RuntimeError(r.stderr[-1500:])
    return r.stdout

def encoder_sei(path, nbytes=4 * 2**20):
    # x264/x265는 파일 앞부분에 옵션 문자열을 남김 (카메라 칩 인코더는 대부분 안 남김)
    b = open(path, "rb").read(nbytes)
    m = re.search(rb"(x26[45] - core[^\x00]{0,3000})", b) or re.search(rb"(x265 \(build[^\x00]{0,3000})", b)
    return m.group(1).decode("latin1") if m else ""

def opt(sei, key):
    m = re.search(rf"(?:^| ){key}=([^ ]+)", sei)
    return m.group(1) if m else None

def analyze(path):
    js = json.loads(sh(f'ffprobe -v error -select_streams v:0 -show_entries '
        f'stream=codec_name,profile,level,width,height,pix_fmt,r_frame_rate,avg_frame_rate,has_b_frames,bit_rate '
        f'-show_entries format=duration,size,bit_rate:format_tags -of json "{path}"'))
    s, fm = js["streams"][0], js["format"]
    n, d = map(float, s["avg_frame_rate"].split("/")); fps = n / d if d else float("nan")
    # 패킷 단위 (디코딩 없이 빠름): 크기, 키프레임, 시간
    pk = json.loads(sh(f'ffprobe -v error -select_streams v:0 -show_entries packet=pts_time,dts_time,size,flags -of json "{path}"'))["packets"]
    size = np.array([int(p["size"]) for p in pk])
    key = np.array(["K" in p.get("flags", "") for p in pk])
    pts = np.array([float(p.get("pts_time", "nan")) for p in pk])
    kidx = np.where(key)[0]
    gop = np.diff(kidx) if len(kidx) > 1 else np.array([len(pk)])
    reorder = bool(np.any(np.diff(pts[~np.isnan(pts)]) < 0))  # 표시 순서 != 디코딩 순서 → B프레임
    # 초당 비트레이트 (마지막 불완전 1초 제외)
    t0 = np.nanmin(pts); sec = np.floor(pts - t0).astype(int)
    per_sec = np.bincount(sec[sec >= 0], weights=size[sec >= 0]) * 8 / 1000
    per_sec = per_sec[:-1] if len(per_sec) > 2 else per_sec
    # I/P/B 비율 (앞 20초만 디코딩)
    pt = sh(f'ffprobe -v error -select_streams v:0 -read_intervals %+20 -show_entries frame=pict_type -of csv=p=0 "{path}"')
    pt = [x.strip().strip(",") for x in pt.split() if x.strip().strip(",")]
    dur = len(pk) / fps if fps == fps and fps > 0 else float(fm["duration"])
    kbps = size.sum() * 8 / dur / 1000
    sei = encoder_sei(path)
    return dict(
        codec=s["codec_name"], profile=s.get("profile"), level=s.get("level"),
        w=s["width"], h=s["height"], pix_fmt=s.get("pix_fmt"), fps=round(fps, 2),
        dur_s=round(dur, 1), kbps=round(kbps, 1), bpp=round(kbps * 1000 / (s["width"] * s["height"] * fps), 4),
        has_b_frames=s.get("has_b_frames"), reorder=reorder,
        I=pt.count("I"), P=pt.count("P"), B=pt.count("B"),
        gop_med=int(np.median(gop)), gop_min=int(gop.min()), gop_max=int(gop.max()), n_key=len(kidx),
        gop_sec=round(float(np.median(gop)) / fps, 2) if fps else None,
        sec_kbps_cv=round(float(per_sec.std() / per_sec.mean()), 3) if len(per_sec) > 1 else None,
        sec_kbps_max_over_mean=round(float(per_sec.max() / per_sec.mean()), 2) if len(per_sec) > 1 else None,
        sec_kbps_p95_over_mean=round(float(np.percentile(per_sec, 95) / per_sec.mean()), 2) if len(per_sec) > 1 else None,
        encoder_tag=(fm.get("tags") or {}).get("encoder"),
        x26x=("x265" if "x265" in sei else "x264" if "x264" in sei else ""),
        sei_rc=opt(sei, "rc"), sei_bitrate=opt(sei, "bitrate"), sei_vbv_maxrate=opt(sei, "vbv-maxrate") or opt(sei, "vbv_maxrate"),
        sei_crf=opt(sei, "crf"), sei_bframes=opt(sei, "bframes"), sei_keyint=opt(sei, "keyint"),
        sei=sei[:600])

In [ ]:
rows = []
for i, (z, zf, m) in enumerate(sample):
    v = vid(m); p = os.path.join(TMP, os.path.basename(m.filename))
    try:
        with zf.open(m) as src, open(p, "wb") as dst:
            shutil.copyfileobj(src, dst, 8 * 2**20)
        r = analyze(p); r.update(video_id=v, zip=z, file_mb=round(m.file_size / 1e6, 1))
        if meta is not None and "VIDEO_ID" in meta:
            mm = meta[meta["VIDEO_ID"].astype(str) == v]
            if len(mm):
                for c in ["SCENARIO", "WEATHER", "DAYTIME", "SCALE", "ROADTYPE"]:
                    if c in mm: r[c] = mm.iloc[0][c]
        rows.append(r)
        print(f"[{i+1}/{len(sample)}] {v}: {r['codec']} {r['profile']} {r['w']}x{r['h']} {r['fps']}fps "
              f"{r['kbps']}kbps bpp {r['bpp']} B={r['B']} GOP {r['gop_med']}({r['gop_sec']}s) CV {r['sec_kbps_cv']} {r['x26x']} {r['sei_rc'] or ''}")
    except Exception as e:
        print("실패", v, str(e)[:300])
    finally:
        if os.path.exists(p): os.remove(p)

df = pd.DataFrame(rows)
df.to_csv(f"{OUT}/tsbow_encoding.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
df.drop(columns=["sei"])

In [ ]:
# 요약
print("영상 수", len(df))
for c in ["codec", "profile", "pix_fmt", "x26x", "sei_rc", "encoder_tag"]:
    print(f"{c:12s}", dict(df[c].fillna("-").value_counts()))
print("해상도      ", dict((df.w.astype(str) + "x" + df.h.astype(str)).value_counts()))
print("fps         ", dict(df.fps.value_counts()))
print("B프레임 있음 ", int(((df.B > 0) | df.reorder).sum()), "/", len(df))
for c in ["kbps", "bpp", "gop_med", "gop_sec", "sec_kbps_cv", "sec_kbps_max_over_mean"]:
    x = df[c].astype(float)
    print(f"{c:22s} 중앙 {x.median():.4g}  최소 {x.min():.4g}  최대 {x.max():.4g}")
if df.sei.str.len().gt(0).any():
    print("\n인코더 옵션 예시:\n", df.loc[df.sei.str.len().gt(0), "sei"].iloc[0])

## (선택) 우리 CCTV형 인코딩과 같은 기준으로 비교
UVG 재실험 노트북 Output(`cctv_enc`)을 Input으로 붙였을 때만 실행. 같은 함수로 분석해서 표에 나란히 놓고 비교함

In [ ]:
import glob
ours = sorted(glob.glob("/kaggle/input/**/cctv_enc/*/cctv_*k.mp4", recursive=True))
print(len(ours), "개")
orows = []
for p in ours:
    if "HighwayView" not in p and "CampusView" not in p: continue
    r = analyze(p); r.update(video_id=os.path.basename(os.path.dirname(p)) + "/" + os.path.basename(p)); orows.append(r)
if orows:
    od = pd.DataFrame(orows); od.to_csv(f"{OUT}/ours_encoding.csv", index=False)
    cols = ["video_id", "codec", "profile", "fps", "kbps", "bpp", "B", "gop_med", "gop_sec", "sec_kbps_cv", "sec_kbps_max_over_mean", "x26x", "sei_rc"]
    display(od[cols])